In [ ]:
"""
In this notebook, we consider a simple example of normalizing flows, based on the free-form formulation in, 

Draxler, F., Sorrenson, P., Zimmermann, L., Rousselot, A., & Köthe, U. (2024, April). 
Free-form flows: Make any architecture a normalizing flow. 
In International Conference on Artificial Intelligence and Statistics (pp. 2197-2205). PMLR.

As documented in the reference, the advantage of this formulation is that it avoids the need to explicitly use
invertible neural networks. Rather, this is enforced through a penalty that promotes reconstruction.

In the following code, we will use only basic jax code as a learning exersize, akin to

https://jax.readthedocs.io/en/latest/notebooks/neural_network_with_tfds_data.html

Edit: 2/11/2025
incorporated AdamW and some other minor adjustments. 
training is still quite unstable 
some debugging seems necessary, training loss behavior is deeply suspicious, but the results are not terrible. 
"""

In [2]:
import time
from typing import Callable
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import numpy as np

In [ ]:
jax.devices()
key = jax.random.key(0)

In [4]:
# settings
NUMBER_OF_EPOCHS = 1200
BATCH_SIZE = 512
DATA_SIZE = 10000
LEARNING_RATE = 3e-4
BETA = 1e2

In [5]:
# dataset

def generate_dataset(n: int = 1000, r_inner = 0.5, r_outer = 1.5):
    #theta = np.linspace(0, 2 * np.pi, n)
    theta = np.random.rand(n) * 2 * np.pi
    r = r_inner + (r_outer - r_inner) / 2 * (1 + np.sin(2 * theta))
    noise = 0.1 * (2.0 * np.random.rand(n) - 1.0)
    x = r * np.cos(theta) * (1 + noise)
    y = r * np.sin(theta) * (1 + noise)

    return np.stack([x, y], axis=1)


train_data = jnp.array( generate_dataset(n=DATA_SIZE) )
valid_data = jnp.array( generate_dataset(n=100) )

In [ ]:
plt.scatter(train_data[:,0], train_data[:,1])

In [7]:

def random_layer_params(
        key: jax.random.key,
        in_dim: int,
        out_dim: int,
        b_scale: float = 0.0,
) -> tuple[jnp.array]:
    
    w_key, b_key = jax.random.split(key, 2)
    return  (
        jax.random.uniform(
           w_key,
           shape=(out_dim, in_dim),
           minval=-jnp.sqrt(1/in_dim),
           maxval=jnp.sqrt(1/in_dim)
        ),
      b_scale * jax.random.uniform(
           b_key,
           shape=(out_dim,),
           minval=-jnp.sqrt(1/in_dim),
           maxval=jnp.sqrt(1/in_dim),
        ),
    )

# Initialize all layers for a fully-connected neural network with sizes "sizes"
def init_network_params(key: jax.random.key, layer_sizes: list[int], b_scale: int = 0.0) -> list[int]:
    keys = jax.random.split(key, len(layer_sizes))
    return [
        random_layer_params(k, in_dim, out_dim, b_scale) 
        for in_dim, out_dim, k in zip(layer_sizes[:-1], layer_sizes[1:], keys)
    ]

In [8]:

# model prediction
# empirically, including layer-wise residual connections (e.g., h = h + activation(h)) results in the loss
# blowing up, leading to nans. The trainability (with just SGD) of this loss is quite sensitive to the network characteristics
# and the network/loss hyperparameters.
def predict_single(params: jnp.array, x: jnp.array) -> jnp.array:
    def relu(z: jnp.array):
        return jnp.maximum(0, z)
    def tanh(z: jnp.array):
        return jnp.tanh(z)
    def sigmoid(z: jnp.array):
        return 1/(1 + jnp.exp(-z))
    def selu(z: jnp.array):
        return z * sigmoid(z)
    
    h = x
    readin_w, readin_b = params[0]
    u = jnp.dot(readin_w, h) + readin_b
    h = relu(u)

    for (w,b) in params[1:-1]:
        u = jnp.dot(w, h) + b
        h = 0.6*h + 0.4*relu(u) 
        # h = tanh(u)
    
    readout_w, readout_b = params[-1]
    return jnp.dot(readout_w, h) + readout_b

In [9]:

def loss_components_single(
    params: list[jnp.array],
    x: jnp.array,
    v: jnp.array
):
    """
    Free-form flow (fff) loss function, as defined in Algorithm 1 of the
    primary reference and derived in Appendix A.2. 
    """

    # note, x is assumed of dimension (d,) 
    # also, jacobian calculations are wrt inputs (not parameters)
    
    encoder_depth = len(params) // 2

    def encoder_fn(x1: jnp.array):
       return predict_single(params[:encoder_depth], x1)
    
    def decoder_fn(z1: jnp.array):
        return predict_single(params[encoder_depth:], z1)
    
    z, func_vjp = jax.vjp(encoder_fn, x)
    v1 = func_vjp(v)[0]
    xr, v2 = jax.jvp(decoder_fn, [z,], [v,])
 
    # v, z, v1, xr, v2 are all lists containing a (B, 1, d) array
    log_jac_det = jax.lax.stop_gradient(v2) * v1
    nll = 0.5 * jnp.square(z).sum(axis=-1) - log_jac_det.sum(axis=-1)
    L_reconstr = jnp.square(xr - x).sum(axis=-1)

    return nll, L_reconstr

# define vmap over rng keys and batches
batch_loss_components = jax.vmap(loss_components_single, in_axes=(None, 0, 0))

def batch_loss(
    params: list[jnp.array],
    x: jnp.array,
    v: jnp.array,
):
    nll, L_reconstr = batch_loss_components(params, x, v)
    return (nll + BETA * L_reconstr).mean()

In [10]:
## OPTIMIZATION

def moving_average_step(
    past_state: list[jnp.array],
    new_state: list[jnp.array],
    beta: float = 0.9,
) -> list[jnp.array]:
    return [
        (beta * w_ps + (1-beta) * w_ns, beta * b_ps +(1-beta) * b_ns)
        for (w_ps, b_ps), (w_ns, b_ns) in zip(past_state, new_state)
    ]
    
def clip_grads(grads: list[jnp.array], clip_threshold: float = 1.0) -> list[jnp.array]:
     return [(_clip(w, clip_threshold), _clip(b, clip_threshold)) for (w, b) in grads]
    
def _clip(g: jnp.array, clip_threshold: float, eps: float = 1e-7) -> jnp.array:
    g_norm = jnp.max(jnp.array([jnp.linalg.norm(g), eps]))
    return jnp.min(jnp.array([clip_threshold/g_norm, 1.0])) * g

def gradient_step(
    params: list[jnp.array],
    grads: list[jnp.array],
    lr: float,
    weight_decay: float = 0,
) -> list[jnp.array]:
    
    return [
        ( (1 - lr * weight_decay) * w - lr * dw, (1 - lr * weight_decay) * b - lr * db) for (w, b), (dw, db) in zip(params, grads)
    ] 

def adamw_step(
    params: list[jnp.array],
    momentum: list[jnp.array],
    velocity: list[jnp.array],
    lr: float,
    weight_decay: float = 1e-4,
    eps: float = 1e-8,
) -> list[jnp.array]:
    
    adjusted_grads = [(w_g / (jnp.sqrt(w_v) + eps), b_g / (jnp.sqrt(b_v) + eps)) for (w_g, b_g), (w_v, b_v) in zip(momentum, velocity)]
    return [
        ( (1 - weight_decay) * w - lr * dw, (1 - weight_decay) * b - lr * db) for (w, b), (dw, db) in zip(params, adjusted_grads)
    ] 

def cosine_lr_scheduler(min_lr: float, max_lr: float, current_epoch: int, epochs_per_cycle: int, decay_rate: float = 1.0) -> float:
     adjusted_max_lr = decay_rate**current_epoch * max_lr
     return min_lr + 0.5 * (adjusted_max_lr - min_lr) * (1 + jnp.cos( (current_epoch % epochs_per_cycle) / epochs_per_cycle * jnp.pi))

def decay_lr_scheduler(lr: float, decay_rate: float = 0.95) -> float:
     return lr * decay_rate

In [ ]:
# networks
layer_sizes = [2, 128, 128, 128, 128, 2]
b_scale = 1.0

key, subkey1, subkey2 = jax.random.split(key, 3)
encoder_params = init_network_params(subkey1, layer_sizes, b_scale)
decoder_params = init_network_params(subkey2, layer_sizes, b_scale)
params = encoder_params + decoder_params

# training loop

train_losses = []
valid_losses = []

lr = LEARNING_RATE
scale = 0.999999
lr_min = 1e-5

# optimization
beta1 = 0.9
beta2 = 0.999
weight_decay = 1e-10

momentum = [(jnp.zeros_like(w), jnp.zeros_like(b)) for w,b in params]
velocity = [(jnp.zeros_like(w), jnp.zeros_like(b)) for w,b in params]

value_and_grad_fn = jax.value_and_grad(batch_loss, argnums=0)

@jax.jit
def update(
    params: list[jnp.array],
    x: jnp.array,
    v: jnp.array,
    momentum: list[jnp.array],
    velocity: list[jnp.array],
    lr: float,
    epoch: int,
)->tuple:
    
    loss_val, grads = value_and_grad_fn(params, x, v)
    clipped_grads = clip_grads(grads, 1.0)

    # adamW, based on:
    # Loshchilov, I., & Hutter, F. (2017). Fixing weight decay regularization in adam.
    # arXiv preprint arXiv:1711.05101, 5.

    momentum = moving_average_step(momentum, clipped_grads, beta1)
    velocity = moving_average_step(velocity, [(w**2, b**2) for w, b in clipped_grads], beta2)
    
    # bias corrections for moving average initializations
    mhat = [(w / (1-beta1**(epoch+1)),  b / (1-beta1**(epoch+1))) for w, b in momentum]
    vhat = [(w / (1-beta2**(epoch+1)),  b / (1-beta2**(epoch+1))) for w, b in velocity]

    return loss_val, adamw_step(params, mhat, vhat, lr, weight_decay, eps=1e-7), momentum, velocity


def batch_iterate(key: jax.random.key, batch_size: int, x: jnp.array):
    perm = jnp.array(jax.random.permutation(key, x.shape[0]))
    for s in range(0, x.shape[0], batch_size):
        ids = perm[s : s + batch_size]
        yield x[ids]

for epoch in range(NUMBER_OF_EPOCHS):
    key, subkey1 = jax.random.split(key, 2)
    start_time = time.time()
    for xb in batch_iterate(subkey1, BATCH_SIZE, train_data):
        key, subkey = jax.random.split(key, 2)

        # Hutchinson trace approximation works better with test vectors v on \sqrt{dim}-sphere.
        # https://www.ethanepperly.com/index.php/2024/01/28/dont-use-gaussians-in-stochastic-trace-estimation/
        #
        # following the main reference, we will just use a single test vector
        vb = jax.random.normal(subkey, shape=xb.shape)
        vb *= jnp.sqrt(vb.shape[-1]) / jnp.sqrt(jnp.square(vb).sum(axis=-1, keepdims=True))

        loss_val, params, momentum, velocity = update(
            params,
            xb,
            vb,
            momentum,
            velocity,
            lr,
            epoch,
        )

    
    # full training loss
    key, subkey = jax.random.split(key, 2)
    v = jax.random.normal(subkey, shape=train_data.shape)
    v *= jnp.sqrt(v.shape[-1]) / jnp.sqrt(jnp.square(v).sum(axis=-1, keepdims=True))

    train_nll, train_reconstr = batch_loss_components(
        params,
        train_data,
        v,
    )
    train_nll = train_nll.mean()
    train_reconstr = train_reconstr.mean()
    train_loss = train_nll + BETA * train_reconstr

    # full validation loss
    key, subkey = jax.random.split(key, 2)
    v = jax.random.normal(subkey, shape=valid_data.shape)
    v *= jnp.sqrt(v.shape[-1]) / jnp.sqrt(jnp.square(v).sum(axis=-1, keepdims=True))

    valid_nll, valid_reconstr = batch_loss_components(
        params,
        valid_data,
        v,
    )
    valid_nll = valid_nll.mean()
    valid_reconstr = valid_reconstr.mean()
    valid_loss = valid_nll + BETA * valid_reconstr
    
    train_losses.append(train_loss.item())
    valid_losses.append(valid_loss.item())
        
    epoch_time = time.time() - start_time
    print(f"Epoch {epoch} | lr: {lr:.2e} | Train nll, reconstr, loss: {train_nll.item():.3f}, {train_reconstr.item():.3f}, {train_loss.item():.3f} | Valid loss {valid_loss.item():.3f} | Epoch time {epoch_time:.4f}s")
    lr = max(lr*scale, lr_min)

In [ ]:
plt.plot(train_losses[10:], c='b', label='train')
plt.plot(valid_losses[10:], c='r', label='valid')
plt.yscale('symlog')

In [13]:
batch_predict = jax.vmap(predict_single, in_axes=(None, 0))

In [ ]:
# generation plot

encoder_depth = len(params) // 2

encoder_params = params[:encoder_depth]
decoder_params = params[encoder_depth:]

num_samples = 1000
key, subkey = jax.random.split(key, 2)
z = jax.random.normal(subkey, shape=(num_samples, 2))
x_gen = batch_predict(decoder_params, z)

plt.scatter(x_gen[:, 0], x_gen[:, 1], c='blue', marker='.', label='decoded/generated')
plt.scatter(z[:, 0], z[:, 1], marker='o', facecolors='none', edgecolors='black', alpha=0.2, label='latents')
plt.scatter(train_data[:, 0], train_data[:, 1], c='red', marker='.', alpha = 1e-2, label='data')
leg = plt.legend()
for lhandle in leg.legend_handles: 
    lhandle.set_alpha(1)

In [ ]:
x_reconstr =  batch_predict(decoder_params, x_gen)
plt.scatter(x_reconstr[:, 0], x_reconstr[:, 1], c='blue', marker='.', label='reconstructed')
plt.scatter(z[:, 0], z[:, 1], marker='o', facecolors='none', edgecolors='black', alpha=0.2, label='latents')
plt.scatter(train_data[:, 0], train_data[:, 1], c='red', marker='.', alpha = 1e-2, label='data')
leg = plt.legend()
for lhandle in leg.legend_handles: 
    lhandle.set_alpha(1)